In [0]:
# ============================================
# NOTEBOOK 1: Data Ingestion & Exploration
# (Compatible con Serverless)
# ============================================

from pyspark.sql.functions import col, count, min, max, avg, when, year, month
from pyspark.sql.functions import round as spark_round
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import to_date

print("="*70)
print("CARGA DE DATOS")
print("="*70)


In [0]:
df1 = spark.table("workspace.default.price_paid_2023_1")
df2 = spark.table("workspace.default.price_paid_2023_2")
df3 = spark.table("workspace.default.price_paid_2024")
df4 = spark.table("workspace.default.price_paid_2025")

In [0]:
# ============================================
#  Carga + renombrado + tipos
# ============================================

df = df1.union(df2).union(df3).union(df4)

columns = [
    "transaction_id", "price", "date_of_transfer", "postcode",
    "property_type", "old_new", "duration", "paon", "saon",
    "street", "locality", "town_city", "district", "county",
    "ppd_category_type", "record_status"
]

df = df.toDF(*columns)

# Cast tipos
df = df.withColumn("price", col("price").cast(DoubleType())) \
       .withColumn("date_of_transfer", to_date(col("date_of_transfer")))

print(f"✓ Registros: {df.count():,}")
print(f"✓ Columnas: {len(df.columns)}")

display(df.limit(5))

In [0]:
# ============================================
# EDA básico
# ============================================

display(df.select("price").describe())

display(df.select(
    min("price").alias("min"),
    max("price").alias("max"),
    avg("price").alias("avg")
))

In [0]:
# ============================================
# Tipos de propiedad
# ============================================

display(df.groupBy("property_type").count().orderBy("count", ascending=False))

In [0]:
# ============================================
# Año
# ============================================

df = df.withColumn("year", year("date_of_transfer"))

display(df.groupBy("year").count().orderBy("year"))

In [0]:
# ============================================
# Guardar Bronze 
# ============================================

df.write.format("delta").mode("overwrite").saveAsTable("bronze_property_sales")

print("Tabla Bronze creada")

display(spark.sql("SELECT COUNT(*) FROM bronze_property_sales"))